# Integer Program

Creating the basic framework of an integer program that will optimise the amount of seats won

In [1]:
import geopandas as gpd
import pandas as pd
import shapely as sp

## Reading in data

In [2]:
nsw_data = gpd.read_file("../data/20190518/act_calculated.geojson")
nsw_data.head()

,State,DivisionID,DivisionNm,PollingPlaceID,PollingPlaceTypeID,PollingPlaceNm,PremisesNm,PremisesAddress1,PremisesAddress2,PremisesAddress3,PremisesSuburb,PremisesStateAb,PremisesPostCode,Latitude,Longitude,geometry
0,ACT,318,Bean,11877,1,Bonython,Bonython Primary School,64 Hurtle Ave,None,None,BONYTHON,ACT,2905.0,-35.431800,149.083000,"POLYGON ((149.089 -35.434, 149.071 -35.445, 14..."
1,ACT,318,Bean,11452,1,Calwell,Calwell High School,111 Casey Cres,None,None,CALWELL,ACT,2905.0,-35.440670,149.117600,"POLYGON ((149.139 -35.433, 149.139 -35.433, 14..."
2,ACT,318,Bean,8761,1,Chapman,Chapman Primary School,46-50 Perry Dr,None,None,CHAPMAN,ACT,2611.0,-35.356400,149.042000,"POLYGON ((148.774 -35.451, 148.773 -35.449, 14..."
3,ACT,318,Bean,8763,1,Chisholm,Caroline Chisholm School,108 Hambidge Cres,None,None,CHISHOLM,ACT,2905.0,-35.419522,149.122539,"POLYGON ((149.124 -35.411, 149.139 -35.433, 14..."
4,ACT,318,Bean,93916,1,City (Bean),Pilgrim House,69 Northbourne Ave,None,None,CANBERRA CITY,ACT,2601.0,-35.276702,149.129081,"POLYGON ((149.112 -35.29, 149.096 -35.28, 149...."


In [3]:
ex_poly = nsw_data["geometry"].iloc[0]
ex_poly1 = nsw_data["geometry"].iloc[1]

In [4]:
adjacency_matrix = [[sp.touches(poly_1, poly_2) 
                     for poly_2 in nsw_data["geometry"]] 
                    for poly_1 in nsw_data["geometry"]]

adjacency_matrix

[[np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.True_,
  np.True_,
  np.False_,
  np.True_,
  np.False_,
  np.False_,
  np.True_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.True_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np.False_,
  np

## Basic integer programming

We have a set of voronoi areas, $v\in V$ and a set of divisions, $d \in D$. A binary variable $a_{rs}$ determines if two regions $r,s\in R$ are adjacent to each other. 
The  binary decision variable $x_{vd}$, $v \in V, d \in D$ determines if region r is included in division d.

Each voronoi polygon can only be selected for one region:

$$
\sum_{d\in D} x_{vd} = 1 \forall  v \in V
$$

Each division must have at least one voronoi polygon

$$
\sum_{v\in V} x_{vd} > 1 \forall  d \in D
$$

# Setting up IP

In [5]:
from ortools.linear_solver import pywraplp
from ortools.graph.python import linear_sum_assignment

# Create the solver (use CBC for IP/MIP)
solver = pywraplp.Solver.CreateSolver('CBC')
# OR use: solver = pywraplp.Solver.CreateSolver('SCIP')

# Create variables
        # binary variable {0,1}
z = solver.NumVar(0, solver.infinity(), 'z')  # continuous variable

### Define variables

In [6]:
num_divisions = 2
booth_list = nsw_data["PollingPlaceID"].unique()
num_voronoi = len(booth_list)

In [7]:

x = {}
for v in range(num_voronoi):
    for d in range(num_divisions):
        x[v, d] = solver.IntVar(0, 1, f"x_{v}_{d}")


### Define constraints

In [8]:
# # Assignment constraints
# for v in range(num_voronoi):
#     solver.Add(solver.sum(x[v,d] for d in range(num_divisions)) == 1)


# for d in range(num_divisions):
#     solver.Add(solver.sum(x[v,d] for v in range(num_voronoi)) > 1)

# Each booth can only be in one division
for v in range(num_voronoi):
    # constraint = solver.RowConstraint(1, 1, "")
    constraint_expr = [x[v, d] for d in range(num_divisions)]
    solver.Add(sum(constraint_expr) == 1)

# There must be at least one booth in each division
for d in range(num_divisions):
    # constraint = solver.RowConstraint(1, 1, "")
    constraint_expr = [x[v, d] for v in range(num_voronoi)]
    solver.Add(sum(constraint_expr) >= 1)
    

### Define objective

In [9]:
objective_terms = []
#starting. withasimpleobjectivefunctiontocheckthatihavetheconstrainsright
for v in range(num_voronoi):
    for d in range(num_divisions):
        objective_terms.append(x[v, d])
        
solver.Minimize(solver.Sum(objective_terms))

In [10]:
status = solver.Solve()


In [11]:
status

0

In [ ]:
if status == pywraplp.Solver.OPTIMAL:
    print("Objective value =", solver.Objective().Value())
   
    print()
    print(f"Problem solved in {solver.wall_time():d} milliseconds")
    print(f"Problem solved in {solver.iterations():d} iterations")
    print(f"Problem solved in {solver.nodes():d} branch-and-bound nodes")
else:
    print("The problem does not have an optimal solution.")

Objective value = 103.0
x_0_0  =  0.0
x_0_1  =  1.0
x_1_0  =  0.0
x_1_1  =  1.0
x_2_0  =  0.0
x_2_1  =  1.0
x_3_0  =  0.0
x_3_1  =  1.0
x_4_0  =  0.0
x_4_1  =  1.0
x_5_0  =  0.0
x_5_1  =  1.0
x_6_0  =  0.0
x_6_1  =  1.0
x_7_0  =  0.0
x_7_1  =  1.0
x_8_0  =  0.0
x_8_1  =  1.0
x_9_0  =  0.0
x_9_1  =  1.0
x_10_0  =  0.0
x_10_1  =  1.0
x_11_0  =  0.0
x_11_1  =  1.0
x_12_0  =  0.0
x_12_1  =  1.0
x_13_0  =  0.0
x_13_1  =  1.0
x_14_0  =  0.0
x_14_1  =  1.0
x_15_0  =  0.0
x_15_1  =  1.0
x_16_0  =  0.0
x_16_1  =  1.0
x_17_0  =  0.0
x_17_1  =  1.0
x_18_0  =  0.0
x_18_1  =  1.0
x_19_0  =  0.0
x_19_1  =  1.0
x_20_0  =  0.0
x_20_1  =  1.0
x_21_0  =  0.0
x_21_1  =  1.0
x_22_0  =  0.0
x_22_1  =  1.0
x_23_0  =  0.0
x_23_1  =  1.0
x_24_0  =  0.0
x_24_1  =  1.0
x_25_0  =  0.0
x_25_1  =  1.0
x_26_0  =  0.0
x_26_1  =  1.0
x_27_0  =  0.0
x_27_1  =  1.0
x_28_0  =  0.0
x_28_1  =  1.0
x_29_0  =  0.0
x_29_1  =  1.0
x_30_0  =  0.0
x_30_1  =  1.0
x_31_0  =  0.0
x_31_1  =  1.0
x_32_0  =  0.0
x_32_1  =  1.0
x_33_0

In [13]:
booth_allocation = pd.Series({polling_place_id:d for v, polling_place_id in enumerate(booth_list) for d in range(num_divisions) if  x[v,d].solution_value()==1})

booth_allocation

11877    1
11452    1
8761     1
8763     1
93916    1
        ..
31626    1
65743    1
8834     1
97559    1
94016    0
Length: 103, dtype: int64

In [14]:
plot_data = nsw_data.set_index("PollingPlaceID", drop=False)
plot_data["division"] = booth_allocation

plot_data

,State,DivisionID,DivisionNm,PollingPlaceID,PollingPlaceTypeID,PollingPlaceNm,PremisesNm,PremisesAddress1,PremisesAddress2,PremisesAddress3,PremisesSuburb,PremisesStateAb,PremisesPostCode,Latitude,Longitude,geometry,division
PollingPlaceID,,,,,,,,,,,,,,,,,
11877,ACT,318,Bean,11877,1,Bonython,Bonython Primary School,64 Hurtle Ave,None,None,BONYTHON,ACT,2905.0,-35.431800,149.083000,"POLYGON ((149.089 -35.434, 149.071 -35.445, 14...",1
11452,ACT,318,Bean,11452,1,Calwell,Calwell High School,111 Casey Cres,None,None,CALWELL,ACT,2905.0,-35.440670,149.117600,"POLYGON ((149.139 -35.433, 149.139 -35.433, 14...",1
8761,ACT,318,Bean,8761,1,Chapman,Chapman Primary School,46-50 Perry Dr,None,None,CHAPMAN,ACT,2611.0,-35.356400,149.042000,"POLYGON ((148.774 -35.451, 148.773 -35.449, 14...",1
8763,ACT,318,Bean,8763,1,Chisholm,Caroline Chisholm School,108 Hambidge Cres,None,None,CHISHOLM,ACT,2905.0,-35.419522,149.122539,"POLYGON ((149.124 -35.411, 149.139 -35.433, 14...",1
93916,ACT,318,Bean,93916,1,City (Bean),Pilgrim House,69 Northbourne Ave,None,None,CANBERRA CITY,ACT,2601.0,-35.276702,149.129081,"POLYGON ((149.112 -35.29, 149.096 -35.28, 149....",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31626,ACT,102,Fenner,31626,1,Palmerston,Palmerston District Primary School,80 Kosciuszko Ave,None,None,PALMERSTON,ACT,2913.0,-35.197500,149.120000,"POLYGON ((149.11 -35.189, 149.132 -35.196, 149...",1
65743,ACT,102,Fenner,65743,1,Parkes (Fenner),Old Parliament House,18 King George Tce,None,None,PARKES,ACT,2600.0,-35.301961,149.129964,"POLYGON ((149.116 -35.301, 149.112 -35.29, 149...",1
8834,ACT,102,Fenner,8834,1,Scullin,Southern Cross Early Childhood School,33 Wirraway Cres,None,None,SCULLIN,ACT,2614.0,-35.232557,149.038403,"POLYGON ((149.055 -35.231, 149.024 -35.241, 14...",1


## Plotting results

In [25]:
import numpy as np
import json
import plotly.express as px
import plotly.graph_objects as go
def plot_voronoi(booths):
    min_lon, min_lat, max_lon, max_lat = booths["geometry"].total_bounds
    centre_lon = (max_lon + min_lon) / 2
    centre_lat = (min_lat + max_lat) / 2
    area = (max_lon - min_lon) * (max_lat - min_lat) * 10
    num_regions = len(booths["division"].unique())
    zoom = np.interp(
        x=area,
        xp=[0, 5**-10, 4**-10, 3**-10, 2**-10, 0.0025, 1**-10, 2**10, 45324, 5**10],
        fp=[20, 17, 16, 15, 14, 11, 7, 5, 3, 1],
    )

    booth_json = json.loads(booths["geometry"].to_json(show_bbox=False))

    fig = px.choropleth_map(
        booths,
        geojson=booth_json,
        locations="PollingPlaceID",
        color="division",
        color_continuous_scale="Viridis",
        range_color=(0, num_regions),
        map_style="carto-positron",
        hover_data=["PremisesNm", "division", "Longitude", "Latitude"],
        opacity=0.5,
        labels={"PollingPlaceID": "Input ID"},
        center={"lat": centre_lat, "lon": centre_lon},
        zoom=zoom,
    )

    fig.add_traces(
        px.scatter_map(
            booths,
            lat="Latitude",
            lon="Longitude",
            hover_name="PremisesNm",
            hover_data=["division", "DivisionNm", "PollingPlaceNm", "PollingPlaceID"],
        ).data
    )

    fig.update_layout(showlegend=False)
    return fig

plot_voronoi(plot_data)